In [1]:
import pandas as pd
from pathlib import Path

raw = Path("../data/raw")
tables = {f.stem: pd.read_csv(f) for f in sorted(raw.glob("*.csv"))}

for name, df in tables.items():
    print(f"{name:45s} {df.shape[0]:>8,} rows  {df.shape[1]:>2} cols")

olist_customers_dataset                         99,441 rows   5 cols
olist_geolocation_dataset                     1,000,163 rows   5 cols
olist_order_items_dataset                      112,650 rows   7 cols
olist_order_payments_dataset                   103,886 rows   5 cols
olist_order_reviews_dataset                     99,224 rows   7 cols
olist_orders_dataset                            99,441 rows   8 cols
olist_products_dataset                          32,951 rows   9 cols
olist_sellers_dataset                            3,095 rows   4 cols
product_category_name_translation                   71 rows   2 cols


In [2]:
for name, df in tables.items():
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    print(f"\n=== {name} ===")
    print("Duplicated rows:", df.duplicated().sum())
    print(nulls if len(nulls) else "No nulls")


=== olist_customers_dataset ===
Duplicated rows: 0
No nulls

=== olist_geolocation_dataset ===
Duplicated rows: 261831
No nulls

=== olist_order_items_dataset ===
Duplicated rows: 0
No nulls

=== olist_order_payments_dataset ===
Duplicated rows: 0
No nulls

=== olist_order_reviews_dataset ===
Duplicated rows: 0
review_comment_title      87656
review_comment_message    58247
dtype: int64

=== olist_orders_dataset ===
Duplicated rows: 0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

=== olist_products_dataset ===
Duplicated rows: 0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

=== olist_sellers_dataset ===
Duplicated rows: 0
No nulls

=== product_category_name_translation

In [3]:
orders = tables["olist_orders_dataset"]
customers = tables["olist_customers_dataset"]

print("order_id unique:", orders.order_id.is_unique)
print("customer_id unique:", customers.customer_id.is_unique)
print("distinct customer_id:", customers.customer_id.nunique())
print("distinct customer_unique_id:", customers.customer_unique_id.nunique())
print()
print(orders.order_status.value_counts())

order_id unique: True
customer_id unique: True
distinct customer_id: 99441
distinct customer_unique_id: 96096

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [4]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
print(orders.order_purchase_timestamp.min(), "->", orders.order_purchase_timestamp.max())
print(orders.groupby(orders.order_purchase_timestamp.dt.to_period("M")).size())

2016-09-04 21:15:19 -> 2018-10-17 17:30:18
order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, dtype: int64


In [5]:
check = pd.crosstab(
    orders.order_status,
    orders.order_delivered_customer_date.isna(),
    rownames=["status"],
    colnames=["delivery_date_missing"],
)
print(check)

delivery_date_missing  False  True 
status                             
approved                   0      2
canceled                   6    619
created                    0      5
delivered              96470      8
invoiced                   0    314
processing                 0    301
shipped                    0   1107
unavailable                0    609


In [8]:
products = tables["olist_products_dataset"]
translation = tables["product_category_name_translation"]
items = tables["olist_order_items_dataset"]
payments = tables["olist_order_payments_dataset"]
reviews = tables["olist_order_reviews_dataset"]

missing_cat = set(products.product_category_name.dropna()) - set(translation.product_category_name)
print("Categories without translation:", missing_cat)
print("Orders without any item:", (~orders.order_id.isin(items.order_id)).sum())
print("Items with unknown product:", (~items.product_id.isin(products.product_id)).sum())
print("Orders with 2+ payments:", (payments.groupby("order_id").size() > 1).sum())
print("Orders with 2+ reviews:", (reviews.groupby("order_id").size() > 1).sum())

Categories without translation: {'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}
Orders without any item: 775
Items with unknown product: 0
Orders with 2+ payments: 2961
Orders with 2+ reviews: 547


In [7]:
orders_cust = orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
per_customer = orders_cust.groupby("customer_unique_id").order_id.nunique()

print("Customers with 2+ orders:", (per_customer >= 2).sum())
print("Repeat purchase rate: {:.2%}".format((per_customer >= 2).mean()))
print(per_customer.value_counts().sort_index())

Customers with 2+ orders: 2997
Repeat purchase rate: 3.12%
order_id
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64


In [9]:
no_items = orders[~orders.order_id.isin(items.order_id)]
print(no_items.order_status.value_counts())

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64
